# 🚀 YOLOv8 Training - Google Colab

This notebook provides a clean training pipeline using our custom YOLOv8 implementation.

## Features
- ✅ Imports from our modular package structure
- ✅ Unified output folder structure (`output/weights`, `output/epoch_samples`, `output/metrics`)
- ✅ Epoch-wise sample visualization
- ✅ Automatic cleanup of old epoch folders (keeps last 5)


## 1️⃣ Setup & Clone Repository


In [ ]:
# Clone the repository
!git clone https://github.com/Gaurav14cs17/YOLOV8-Pytorch.git
%cd YOLOV8-Pytorch

# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q opencv-python albumentations pyyaml tqdm tensorboard gdown


In [ ]:
# Add project to path and verify
import sys
sys.path.insert(0, '.')

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2️⃣ Import Our Packages


In [ ]:
# Import from our modular packages
import os
import csv
import copy
import shutil
from pathlib import Path
from typing import Dict, Optional, List, Tuple

import cv2
import yaml
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm

# Import from our packages
from model import YOLO, yolo_v8_n, yolo_v8_s, yolo_v8_m
from model.fusion import fuse
from utils import (
    setup_seed, setup_multi_processes,
    EMA, AverageMeter,
    ComputeLoss,
    compute_ap, smooth,
    non_max_suppression, scale, box_iou,
    strip_optimizer, clip_gradients
)
from dataloader import Dataset

print("✅ All packages imported successfully!")


## 3️⃣ Output Manager & Epoch Visualizer


In [ ]:
class OutputManager:
    """Manages all training outputs in a single folder structure."""
    
    def __init__(self, base_dir: str = 'output'):
        self.base_dir = Path(base_dir)
        self.weights_dir = self.base_dir / 'weights'
        self.samples_dir = self.base_dir / 'epoch_samples'
        self.metrics_dir = self.base_dir / 'metrics'
        
        # Create all directories
        for d in [self.weights_dir, self.samples_dir, self.metrics_dir]:
            d.mkdir(parents=True, exist_ok=True)
    
    def get_weight_path(self, name: str) -> Path:
        return self.weights_dir / name
    
    def get_metrics_path(self, name: str) -> Path:
        return self.metrics_dir / name


class EpochVisualizer:
    """Saves sample detection images at each epoch and manages folder cleanup."""
    
    def __init__(self, output_dir: Path, max_folders: int = 5):
        self.output_dir = output_dir
        self.max_folders = max_folders
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # Colors for visualization (BGR format)
        np.random.seed(42)
        self.colors = np.random.randint(0, 255, size=(100, 3)).tolist()
    
    def save_sample(self, model: nn.Module, dataloader: DataLoader, 
                   epoch: int, device: torch.device, class_names: Dict[int, str]):
        """Save one sample detection image for the current epoch."""
        epoch_folder = self.output_dir / f'epoch_{epoch:04d}'
        epoch_folder.mkdir(parents=True, exist_ok=True)
        
        model.eval()
        
        for images, targets, _ in dataloader:
            images = images.to(device).float() / 255.0
            
            with torch.no_grad():
                outputs = model(images)
                predictions = non_max_suppression(outputs, conf_threshold=0.25, iou_threshold=0.45)
            
            img_tensor = images[0]
            pred = predictions[0]
            
            img_np = img_tensor.cpu().numpy().transpose(1, 2, 0)
            img_np = (img_np * 255).astype(np.uint8)
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
            
            for det in pred:
                x1, y1, x2, y2, conf, cls_id = det.cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                cls_id = int(cls_id)
                
                color = self.colors[cls_id % len(self.colors)]
                cv2.rectangle(img_np, (x1, y1), (x2, y2), color, 2)
                
                label = f'{class_names.get(cls_id, f"cls_{cls_id}")} {conf:.2f}'
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                cv2.rectangle(img_np, (x1, y1 - th - 5), (x1 + tw, y1), color, -1)
                cv2.putText(img_np, label, (x1, y1 - 5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            
            info_text = f'Epoch {epoch} | Predictions: {len(pred)}'
            cv2.putText(img_np, info_text, (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            save_path = epoch_folder / f'sample_epoch_{epoch:04d}.jpg'
            cv2.imwrite(str(save_path), img_np)
            break
        
        model.train()
        self.cleanup_old_folders()
    
    def cleanup_old_folders(self):
        """Remove old epoch folders, keeping only the most recent max_folders."""
        folders = sorted([
            f for f in self.output_dir.iterdir() 
            if f.is_dir() and f.name.startswith('epoch_')
        ])
        
        while len(folders) > self.max_folders:
            oldest = folders.pop(0)
            shutil.rmtree(oldest)

print("✅ Output Manager and Epoch Visualizer defined!")


## 4️⃣ Configuration


In [ ]:
class Config:
    """Training configuration."""
    
    def __init__(self, config_path: str = 'config/config.yml'):
        with open(config_path, 'r') as f:
            self.data = yaml.safe_load(f)
    
    @property
    def train_path(self) -> str:
        return self.data['data']['train']
    
    @property
    def val_path(self) -> str:
        return self.data['data']['val']
    
    @property
    def class_names(self) -> Dict[int, str]:
        return self.data.get('names', {})
    
    @property
    def num_classes(self) -> int:
        return len(self.class_names)
    
    @property
    def hyp(self) -> dict:
        return self.data.get('hyp', {})
    
    def get(self, key, default=None):
        return self.data.get(key, default)

# Training parameters - EDIT THESE
EPOCHS = 50
BATCH_SIZE = 8
INPUT_SIZE = 640
VARIANT = 'nano'  # Options: nano, small, medium, large, xlarge

print(f"📋 Training config: {EPOCHS} epochs, batch size {BATCH_SIZE}, input size {INPUT_SIZE}")


## 5️⃣ Download Dataset (Optional - Using COCO128)


In [ ]:
# Download COCO128 dataset (small subset for testing)
DATASET_DIR = Path('dataset/coco128')

if not DATASET_DIR.exists():
    print("📥 Downloading COCO128 dataset...")
    !wget -q https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip -O coco128.zip
    !unzip -q coco128.zip -d dataset/
    !rm coco128.zip
    print("✅ Dataset downloaded!")
else:
    print("📁 Dataset already exists")

# Show dataset structure
!echo "📁 Dataset structure:"
!find dataset/coco128 -type d | head -10


## 6️⃣ Trainer Class


In [ ]:
class ColabTrainer:
    """Training engine for Google Colab."""
    
    def __init__(self, config: Config, epochs: int, batch_size: int, 
                 input_size: int, variant: str = 'nano'):
        self.config = config
        self.epochs = epochs
        self.batch_size = batch_size
        self.input_size = input_size
        self.variant = variant
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Output manager (single folder for all outputs)
        self.output = OutputManager('output')
        
        # Initialize components
        self._init_model()
        self._init_data()
        self._init_optimizer()
        self._init_training()
        
        # Epoch visualization
        self.visualizer = EpochVisualizer(
            output_dir=self.output.samples_dir,
            max_folders=5
        )
        
        self.best_map = 0.0
        print(f"✅ Trainer initialized on {self.device}")
    
    def _init_model(self):
        """Initialize model."""
        factory = {
            'nano': yolo_v8_n,
            'small': yolo_v8_s,
            'medium': yolo_v8_m,
        }
        
        create_fn = factory.get(self.variant, yolo_v8_n)
        self.model = create_fn(self.config.num_classes)
        self.model = self.model.to(self.device)
        
        # Try to load pretrained weights
        weight_path = f'weights/v8_{self.variant[0]}.pth'
        if os.path.exists(weight_path):
            state = torch.load(weight_path, map_location=self.device, weights_only=False)
            self.model.load_state_dict(state, strict=False)
            print(f"✅ Loaded pretrained weights: {weight_path}")
        
        # EMA
        self.ema = EMA(self.model)
        
        # Criterion
        self.criterion = ComputeLoss(self.model, self.config.hyp)
        
        print(f"Model: YOLOv8-{self.variant}, Classes: {self.config.num_classes}")
    
    def _init_data(self):
        """Initialize data loaders."""
        train_path = Path(self.config.train_path)
        val_path = Path(self.config.val_path)
        
        formats = ('jpg', 'jpeg', 'png', 'bmp')
        
        train_files = []
        for fmt in formats:
            train_files.extend(train_path.glob(f'*.{fmt}'))
        
        val_files = []
        for fmt in formats:
            val_files.extend(val_path.glob(f'*.{fmt}'))
        
        train_files = [str(f) for f in train_files]
        val_files = [str(f) for f in val_files]
        
        train_dataset = Dataset(
            filenames=train_files,
            input_size=self.input_size,
            params=self.config.hyp,
            augment=True
        )
        
        val_dataset = Dataset(
            filenames=val_files,
            input_size=self.input_size,
            params=self.config.hyp,
            augment=False
        )
        
        self.train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=2,
            collate_fn=Dataset.collate_fn,
            pin_memory=True
        )
        
        self.val_loader = DataLoader(
            val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=2,
            collate_fn=Dataset.collate_fn,
            pin_memory=True
        )
        
        print(f"Train: {len(train_dataset)} samples, Val: {len(val_dataset)} samples")
    
    def _init_optimizer(self):
        """Initialize optimizer."""
        lr0 = self.config.hyp.get('lr0', 0.01)
        momentum = self.config.hyp.get('momentum', 0.937)
        weight_decay = self.config.hyp.get('weight_decay', 0.0005)
        
        self.optimizer = torch.optim.SGD(
            self.model.parameters(),
            lr=lr0,
            momentum=momentum,
            weight_decay=weight_decay,
            nesterov=True
        )
        
        lrf = self.config.hyp.get('lrf', 0.01)
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(
            self.optimizer,
            lr_lambda=lambda x: (1 - x / self.epochs) * (1 - lrf) + lrf
        )
    
    def _init_training(self):
        """Initialize training utilities."""
        self.scaler = GradScaler()
        self.accumulate = max(round(64 / self.batch_size), 1)
    
    def train_epoch(self, epoch: int) -> AverageMeter:
        """Train for one epoch."""
        self.model.train()
        loss_meter = AverageMeter()
        
        pbar = tqdm(enumerate(self.train_loader), total=len(self.train_loader))
        self.optimizer.zero_grad()
        
        for i, (images, targets, _) in pbar:
            images = images.to(self.device).float() / 255.0
            targets = targets.to(self.device)
            
            with autocast():
                outputs = self.model(images)
                loss = self.criterion(outputs, targets)
            
            self.scaler.scale(loss).backward()
            
            if (i + 1) % self.accumulate == 0:
                self.scaler.unscale_(self.optimizer)
                clip_gradients(self.model)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad()
                self.ema.update(self.model)
            
            loss_meter.update(loss.item(), images.size(0))
            
            mem = f'{torch.cuda.memory_reserved() / 1E9:.3f}G' if torch.cuda.is_available() else '0G'
            pbar.set_description(f'Epoch [{epoch+1}/{self.epochs}] Mem: {mem} Loss: {loss_meter.avg:.4f}')
        
        self.scheduler.step()
        return loss_meter
    
    @torch.no_grad()
    def evaluate(self) -> Tuple[float, float]:
        """Evaluate model."""
        model = self.ema.ema
        model.eval()
        
        iou_v = torch.linspace(0.5, 0.95, 10).to(self.device)
        n_iou = iou_v.numel()
        
        stats = []
        
        for images, targets, shapes in tqdm(self.val_loader, desc='Evaluating'):
            images = images.to(self.device).float() / 255.0
            targets = targets.to(self.device)
            
            outputs = model(images)
            outputs = non_max_suppression(outputs, conf_threshold=0.001, iou_threshold=0.65)
            
            for i, output in enumerate(outputs):
                idx = targets[:, 0] == i
                cls = targets[idx, 1]
                bbox = targets[idx, 2:]
                nl = len(cls)
                
                if len(output) == 0:
                    if nl:
                        stats.append((torch.zeros(0, n_iou, dtype=torch.bool),
                                    torch.Tensor(), torch.Tensor(), cls))
                    continue
                
                pred_cls = output[:, 5]
                pred_box = output[:, :4]
                
                correct = torch.zeros(len(output), n_iou, dtype=torch.bool, device=self.device)
                
                if nl:
                    from utils.bbox import wh2xy
                    target_box = wh2xy(bbox, images.shape[3], images.shape[2], 0, 0)
                    
                    iou = box_iou(target_box, pred_box)
                    correct_class = cls[:, None] == pred_cls
                    
                    for j, iou_thresh in enumerate(iou_v):
                        x = torch.where((iou >= iou_thresh) & correct_class)
                        if x[0].shape[0]:
                            matches = torch.cat((torch.stack(x, 1), iou[x[0], x[1]][:, None]), 1)
                            if x[0].shape[0] > 1:
                                matches = matches[matches[:, 2].argsort(descending=True)]
                                matches = matches[torch.unique(matches[:, 1], return_index=True)[1]]
                                matches = matches[torch.unique(matches[:, 0], return_index=True)[1]]
                            correct[matches[:, 1].long(), j] = True
                
                stats.append((correct.cpu(), output[:, 4].cpu(), pred_cls.cpu(), cls.cpu()))
        
        stats = [torch.cat(x, 0).numpy() for x in zip(*stats)]
        if len(stats) and stats[0].any():
            tp, fp, m_pre, m_rec, map50, mean_ap = compute_ap(*stats)
            return map50, mean_ap
        
        return 0.0, 0.0
    
    def save_checkpoint(self, epoch: int, map50: float, mean_ap: float):
        """Save model checkpoint."""
        model_to_save = self.ema.ema
        checkpoint = {'model': copy.deepcopy(model_to_save).half()}
        
        torch.save(checkpoint, self.output.get_weight_path('last.pt'))
        
        if mean_ap > self.best_map:
            self.best_map = mean_ap
            torch.save(checkpoint, self.output.get_weight_path('best.pt'))
    
    def train(self):
        """Main training loop."""
        metrics_file = self.output.get_metrics_path('step.csv')
        
        with open(metrics_file, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['epoch', 'mAP@50', 'mAP'])
            writer.writeheader()
            
            for epoch in range(self.epochs):
                loss_meter = self.train_epoch(epoch)
                
                map50, mean_ap = self.evaluate()
                print(f"mAP@0.5: {map50:.4f} mAP@0.5:0.95: {mean_ap:.4f}")
                
                # Save sample detection image
                self.visualizer.save_sample(
                    model=self.ema.ema,
                    dataloader=self.val_loader,
                    epoch=epoch + 1,
                    device=self.device,
                    class_names=self.config.class_names
                )
                
                writer.writerow({
                    'epoch': str(epoch + 1).zfill(3),
                    'mAP@50': f'{map50:.3f}',
                    'mAP': f'{mean_ap:.3f}'
                })
                f.flush()
                self.save_checkpoint(epoch, map50, mean_ap)
        
        # Strip optimizer from saved weights
        for name in ['best.pt', 'last.pt']:
            weight_file = self.output.get_weight_path(name)
            if weight_file.exists():
                strip_optimizer(str(weight_file))
        
        torch.cuda.empty_cache()
        print("✅ Training complete!")

print("✅ Trainer class defined!")


## 7️⃣ Start Training


In [ ]:
# Setup
setup_seed()
setup_multi_processes()

# Load config
config = Config('config/config.yml')

# Create trainer
trainer = ColabTrainer(
    config=config,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    input_size=INPUT_SIZE,
    variant=VARIANT
)

# Start training!
trainer.train()


## 8️⃣ View Results


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Show output folder structure
print("📁 Output folder structure:")
!tree output 2>/dev/null || find output -type f | head -20

# Show sample images from last epochs
samples_dir = Path('output/epoch_samples')
epoch_folders = sorted(samples_dir.glob('epoch_*'))

if epoch_folders:
    fig, axes = plt.subplots(1, min(len(epoch_folders), 5), figsize=(20, 4))
    if len(epoch_folders) == 1:
        axes = [axes]
    
    for ax, folder in zip(axes, epoch_folders[-5:]):
        img_files = list(folder.glob('*.jpg'))
        if img_files:
            img = Image.open(img_files[0])
            ax.imshow(img)
            ax.set_title(folder.name)
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No epoch samples found")


In [ ]:
# Plot training metrics
import pandas as pd

metrics_file = Path('output/metrics/step.csv')
if metrics_file.exists():
    df = pd.read_csv(metrics_file)
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.plot(df['epoch'], df['mAP@50'], label='mAP@0.5', marker='o')
    ax.plot(df['epoch'], df['mAP'], label='mAP@0.5:0.95', marker='s')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('mAP')
    ax.set_title('Training Progress')
    ax.legend()
    ax.grid(True)
    plt.show()
else:
    print("No metrics file found")


## 9️⃣ Download Trained Model


In [ ]:
from google.colab import files

# Download best model
best_path = Path('output/weights/best.pt')
if best_path.exists():
    files.download(str(best_path))
    print("✅ Downloaded best.pt")
else:
    print("⚠️ best.pt not found")

# Download last model
last_path = Path('output/weights/last.pt')
if last_path.exists():
    files.download(str(last_path))
    print("✅ Downloaded last.pt")
